In [1]:
# --- IMPORTS & SETUP ---
from __future__ import annotations

import argparse
import json
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Sequence, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
    roc_auc_score,
)
from sklearn.preprocessing import StandardScaler

# TensorFlow y Keras
import tensorflow as tf
from tensorflow.keras import callbacks, layers, models, optimizers

# --- CONFIGURACIÓN DEL PIPELINE (VERSIÓN COMPLETA) ---
@dataclass
class PipelineConfig:
    data_path: str = '../../data/mixed_population_dataset_160_households.csv'
    
    # Parámetros de Secuencia (Ventana de 24h, se mueve cada 15 min)
    sequence_length: int = 96
    sequence_stride: int = 4 # Stride=1 es el más exhaustivo, genera más secuencias
    
    # Parámetros de División de Datos
    train_split_ratio: float = 0.9
    validation_split: float = 0.15 # Un 15% para una validación más robusta
    
    # Hiperparámetros del Modelo (Arquitectura más profunda)
    latent_dim: int = 32
    lstm_units: Sequence[int] = (128, 64) # Capas LSTM más grandes
    
    # Hiperparámetros de Entrenamiento
    batch_size: int = 256
    max_epochs: int = 100 # Más épocas para un aprendizaje más profundo
    patience: int = 10    # Más paciencia para el Early Stopping
    
    # Hiperparámetro de Evaluación
    threshold_percentile: float = 99.0 # Un umbral más estricto

# Crear una instancia de la configuración
config = PipelineConfig()

# Fijar semillas para reproducibilidad
tf.random.set_seed(42)
np.random.seed(42)

print("Configuración 'Pro' y librerías listas.")

Configuración 'Pro' y librerías listas.


In [2]:
def engineer_features(df: pd.DataFrame) -> Tuple[pd.DataFrame, List[str]]:
    """
    Crea un conjunto completo de características temporales y de ventana deslizante.
    """
    print("Iniciando Feature Engineering (Versión Deluxe)...")
    df = df.copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.sort_values(["household_id", "timestamp"]).reset_index(drop=True)

    # --- 1. Características Temporales Cíclicas (más robustas) ---
    # En lugar de solo 'hour', usamos transformaciones seno/coseno para que el modelo
    # entienda que la hora 23 y la hora 0 son contiguas.
    df['hour_sin'] = np.sin(2 * np.pi * df['timestamp'].dt.hour / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['timestamp'].dt.hour / 24)
    
    df['day_of_week_sin'] = np.sin(2 * np.pi * df['timestamp'].dt.dayofweek / 7)
    df['day_of_week_cos'] = np.cos(2 * np.pi * df['timestamp'].dt.dayofweek / 7)
    
    df['is_weekend'] = (df['timestamp'].dt.dayofweek >= 5).astype(int)
    df['is_night'] = df['timestamp'].dt.hour.between(0, 5, inclusive="both").astype(int)

    # --- 2. Características de Ventana Deslizante (más tamaños y estadísticas) ---
    grouped = df.groupby("household_id", sort=False)['consumption_l']
    
    # Añadimos ventanas más largas (12h, 24h) para capturar patrones de más largo plazo
    for window in [4, 8, 12, 24, 48, 96]: # 1h, 2h, 3h, 6h, 12h, 24h
        rolling = grouped.rolling(window=window, min_periods=1)
        
        rolling_mean = rolling.mean().reset_index(level=0, drop=True)
        rolling_std = rolling.std().reset_index(level=0, drop=True)
        
        df[f"rolling_mean_{window}"] = rolling_mean
        df[f"rolling_std_{window}"] = rolling_std
        df[f"rolling_cv_{window}"] = rolling_std / (rolling_mean + 1e-6)
        df[f"rolling_min_{window}"] = rolling.min().reset_index(level=0, drop=True)
        df[f"rolling_max_{window}"] = rolling.max().reset_index(level=0, drop=True)
        # Diferencia entre el consumo actual y la media de la ventana
        df[f'diff_from_rolling_mean_{window}'] = df['consumption_l'] - df[f'rolling_mean_{window}']
        
    df.fillna(0, inplace=True)

    # --- 3. Características de Lag (Comparación con el Pasado) ---
    # Comparar el consumo actual con el de 1 día y 1 semana antes.
    # Muy útil para detectar desviaciones del comportamiento habitual.
    for lag in [96, 96*7]: # 1 día, 1 semana
        df[f'consumption_lag_{lag}'] = grouped.shift(lag).fillna(0)
    
    # ---- Lista Final de Todas las Características ---
    feature_cols = [col for col in df.columns if col not in ['timestamp', 'is_leak', 'household_id']]
    
    print(f"✓ Feature Engineering Deluxe completado. {len(feature_cols)} características creadas.")
    return df, feature_cols

In [3]:
def create_sequences(
    df: pd.DataFrame,
    feature_cols: Sequence[str],
    seq_length: int,
    stride: int,
) -> Tuple[np.ndarray, np.ndarray, List[Dict]]:
    """Generate overlapping sequences and associated labels/metadata."""
    sequences: List[np.ndarray] = []
    labels: List[int] = []
    metadata: List[Dict] = []

    for household_id, group in df.groupby("household_id"):
        group = group.sort_values("timestamp")
        values = group[feature_cols].values
        leak_flags = group["is_leak"].values
        timestamps = group["timestamp"].values

        if len(group) < seq_length:
            continue

        for start in range(0, len(group) - seq_length + 1, stride):
            end = start + seq_length
            seq = values[start:end]
            label = int(leak_flags[start:end].max())

            sequences.append(seq)
            labels.append(label)
            metadata.append(
                {
                    "household_id": household_id,
                    "start_timestamp": timestamps[start],
                    "end_timestamp": timestamps[end - 1],
                }
            )

    if not sequences:
        raise ValueError("No sequences were generated; check sequence length/stride.")

    return np.stack(sequences), np.array(labels, dtype=int), metadata


In [4]:
# ---------------------------------------------------------------------------
# Model definition
# ---------------------------------------------------------------------------

def build_lstm_autoencoder(
    input_shape: Tuple[int, int],
    latent_dim: int,
    lstm_units: Sequence[int],
) -> tf.keras.Model:
    inputs = layers.Input(shape=input_shape)
    x = inputs
    for units in lstm_units:
        x = layers.LSTM(units, return_sequences=True)(x)
        x = layers.Dropout(0.2)(x)
    x = layers.LSTM(lstm_units[-1] // 2, return_sequences=False)(x)
    latent = layers.Dense(latent_dim, activation="relu")(x)

    x = layers.RepeatVector(input_shape[0])(latent)
    for units in reversed(lstm_units):
        x = layers.LSTM(units, return_sequences=True)(x)
        x = layers.Dropout(0.2)(x)
    outputs = layers.TimeDistributed(layers.Dense(input_shape[1]))(x)

    model = models.Model(inputs, outputs, name="lstm_autoencoder")
    model.compile(optimizer=optimizers.Adam(1e-3), loss="mse")
    return model

In [5]:
# ---------------------------------------------------------------------------
# Training & Evaluation
# ---------------------------------------------------------------------------

def train_autoencoder(
    model: tf.keras.Model,
    X_train: np.ndarray,
    X_val: np.ndarray,
    config: PipelineConfig,
) -> tf.keras.callbacks.History:
    es = callbacks.EarlyStopping(
        monitor="val_loss",
        patience=config.patience,
        restore_best_weights=True,
        verbose=1,
    )
    rlr = callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=max(2, config.patience // 2),
        min_lr=1e-5,
        verbose=1,
    )

    history = model.fit(
        X_train,
        X_train,
        validation_data=(X_val, X_val),
        epochs=config.max_epochs,
        batch_size=config.batch_size,
        callbacks=[es, rlr],
        shuffle=True,
        verbose=2,
    )
    return history


def compute_reconstruction_errors(
    model: tf.keras.Model,
    X: np.ndarray,
) -> np.ndarray:
    reconstructions = model.predict(X, verbose=0)
    errors = np.mean(np.square(X - reconstructions), axis=(1, 2))
    return errors


def determine_threshold(errors: np.ndarray, percentile: float) -> float:
    return float(np.percentile(errors, percentile))


def evaluate_sequences(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    errors: np.ndarray,
) -> Dict:
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )
    metrics = {
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "roc_auc": float(roc_auc_score(y_true, errors)),
        "support": int(y_true.sum()),
    }
    metrics["confusion_matrix"] = confusion_matrix(y_true, y_pred).tolist()
    metrics["classification_report"] = classification_report(
        y_true, y_pred, target_names=["normal", "leak"], zero_division=0
    )
    return metrics

In [6]:
# ---------------------------------------------------------------------------
# Main pipeline
# ---------------------------------------------------------------------------

def run_pipeline(config: PipelineConfig) -> Dict:
    print("Loading dataset:", config.data_path.resolve())
    df = pd.read_csv(config.data_path)
    df, feature_cols = engineer_features(df)

    #split_idx = int(len(df) * config.train_split_ratio)
    #split_idx = min(split_idx, len(df) - 1)
    #split_timestamp = df.iloc[split_idx]["timestamp"]
    #print(f"Chronological split at timestamp: {split_timestamp}")

    scaler = StandardScaler()
    df[feature_cols] = scaler.fit_transform(df[feature_cols])
    df[feature_cols] = df[feature_cols].astype(np.float32)

    sequences, labels, metadata = create_sequences(
        df, feature_cols, seq_length=config.sequence_length, stride=config.sequence_stride
    )
    print(f"Generated {len(sequences)} sequences "
          f"({sequences.shape[1]} timesteps, {sequences.shape[2]} features)")

    # Map metadata to chronological split
    end_times = np.array([meta["end_timestamp"] for meta in metadata])
    #train_mask = end_times < split_timestamp

    sorted_indices = np.argsort(end_times)
    split_idx = int(len(end_times) * config.train_split_ratio)
    split_idx = min(max(split_idx, 1), len(end_times) - 1)
    train_mask = np.zeros(len(end_times), dtype=bool)
    train_mask[sorted_indices[:split_idx]] = True
    split_timestamp = end_times[sorted_indices[split_idx - 1]]
    print(f"Chronological split at timestamp: {split_timestamp}")


    X_train_full = sequences[train_mask]
    y_train_full = labels[train_mask]
    X_test = sequences[~train_mask]
    y_test = labels[~train_mask]

    print(f"Train sequences: {len(X_train_full)} | Test sequences: {len(X_test)}")

    # Keep only normal sequences for unsupervised training
    normal_mask = y_train_full == 0
    X_train_normal = X_train_full[normal_mask]
    print(f"Normal sequences for training: {len(X_train_normal)}")
    print(f"Leak ratio (train): {y_train_full.mean():.6f}")

    # Validation split
    val_size = max(1, int(len(X_train_normal) * config.validation_split))

    if len(X_train_normal) <= val_size:
        # Muy pocas secuencias normales: usa todas para entrenar y validar
        X_train = X_train_normal
        X_val = X_train_normal
    else:
        X_val = X_train_normal[-val_size:]
        X_train = X_train_normal[:-val_size]
    
    if len(X_train) == 0:
        raise ValueError("No hay secuencias normales suficientes para entrenar el autoencoder.")

    autoencoder = build_lstm_autoencoder(
        input_shape=(config.sequence_length, len(feature_cols)),
        latent_dim=config.latent_dim,
        lstm_units=config.lstm_units,
    )
    autoencoder.summary()

    train_autoencoder(autoencoder, X_train, X_val, config)

    val_errors = compute_reconstruction_errors(autoencoder, X_val)
    threshold = determine_threshold(val_errors, config.threshold_percentile)
    print(f"Threshold (p{config.threshold_percentile}): {threshold:.6f}")

    test_errors = compute_reconstruction_errors(autoencoder, X_test)
    y_pred = (test_errors >= threshold).astype(int)
    metrics = evaluate_sequences(y_test, y_pred, test_errors)

    print("\n=== Evaluation ===")
    print(f"Precision: {metrics['precision']:.4f}")
    print(f"Recall   : {metrics['recall']:.4f}")
    print(f"F1-score : {metrics['f1']:.4f}")
    print(f"ROC-AUC  : {metrics['roc_auc']:.4f}")
    print(metrics["classification_report"])
    print("Confusion matrix:", metrics["confusion_matrix"])

    results = {
        "config": vars(config),
        "threshold": threshold,
        "val_error_stats": {
            "mean": float(val_errors.mean()),
            "std": float(val_errors.std()),
            "percentiles": {
                "95": float(np.percentile(val_errors, 95)),
                "99": float(np.percentile(val_errors, 99)),
            },
        },
        "metrics": metrics,
    }
    return results


def parse_args() -> PipelineConfig:
    parser = argparse.ArgumentParser(description="Train LSTM Autoencoder for leak detection.")
    parser.add_argument("--data-path", type=Path, default=None, help="Path to CSV dataset.")
    parser.add_argument("--sequence-length", type=int, default=None, help="Sequence length (timesteps).")
    parser.add_argument("--sequence-stride", type=int, default=None, help="Stride between sequences.")
    parser.add_argument("--latent-dim", type=int, default=None, help="Latent dimension size.")
    parser.add_argument("--batch-size", type=int, default=None, help="Batch size.")
    parser.add_argument("--max-epochs", type=int, default=None, help="Max training epochs.")
    parser.add_argument("--threshold-percentile", type=float, default=None, help="Percentile for threshold.")

    args = parser.parse_args()
    config = PipelineConfig()

    if args.data_path:
        config.data_path = args.data_path
    if args.sequence_length:
        config.sequence_length = args.sequence_length
    if args.sequence_stride:
        config.sequence_stride = args.sequence_stride
    if args.latent_dim:
        config.latent_dim = args.latent_dim
    if args.batch_size:
        config.batch_size = args.batch_size
    if args.max_epochs:
        config.max_epochs = args.max_epochs
    if args.threshold_percentile:
        config.threshold_percentile = args.threshold_percentile

    return config


if __name__ == "__main__":
    tf.random.set_seed(42)
    np.random.seed(42)

    config = PipelineConfig(
    data_path=Path("../../data/mixed_population_dataset_160_households.csv"),
    sequence_length=96,      # 24 h (96 intervalos de 15 min)
    sequence_stride=4,       # mueve la ventana cada intervalo; sube a 4 si pesa mucho
    validation_split=0.15,   # usa 15% de las secuencias normales para validación
    train_split_ratio=0.9,   # 80% inicial del timeline para entrenar, 20% para test
    latent_dim=32,
    lstm_units=(128, 64),
    batch_size=256,
    max_epochs=100,
    patience=10,
    threshold_percentile=99.0
    )

    outputs = run_pipeline(config)

    output_path = Path("lstm_autoencoder_results.json")
    output_path.write_text(json.dumps(outputs, indent=2))
    print(f"\n✓ Results saved to {output_path.resolve()}")

Loading dataset: /Users/sotoa/Documents/TFM/Repository/TFM/backend/data/mixed_population_dataset_160_households.csv
Iniciando Feature Engineering (Versión Deluxe)...
✓ Feature Engineering Deluxe completado. 45 características creadas.
Generated 341920 sequences (96 timesteps, 45 features)
Chronological split at timestamp: 2026-01-23T02:45:00.000000000
Train sequences: 307728 | Test sequences: 34192
Normal sequences for training: 305134
Leak ratio (train): 0.008430


Model: "lstm_autoencoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 96, 45)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 96, 128)        │        89,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 96, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 96, 64)         │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 96, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector (RepeatVector)    │ (None, 96, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 96, 64)         │        24,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 96, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ (None, 96, 128)        │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 96, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 96, 45)         │         5,805 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 281,421 (1.07 MB)

 Trainable params: 281,421 (1.07 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
1014/1014 - 930s - 917ms/step - loss: 0.5516 - val_loss: 0.7078 - learning_rate: 0.0010
Epoch 2/100
1014/1014 - 914s - 902ms/step - loss: 0.4305 - val_loss: 0.6151 - learning_rate: 0.0010
Epoch 3/100
1014/1014 - 922s - 909ms/step - loss: 0.3881 - val_loss: 0.5907 - learning_rate: 0.0010
Epoch 4/100
1014/1014 - 933s - 920ms/step - loss: 0.3660 - val_loss: 0.5553 - learning_rate: 0.0010
Epoch 5/100
1014/1014 - 952s - 939ms/step - loss: 0.3520 - val_loss: 0.5127 - learning_rate: 0.0010
Epoch 6/100
1014/1014 - 965s - 952ms/step - loss: 0.3434 - val_loss: 0.6284 - learning_rate: 0.0010
Epoch 7/100
1014/1014 - 969s - 956ms/step - loss: 0.3389 - val_loss: 0.4522 - learning_rate: 0.0010
Epoch 8/100
1014/1014 - 962s - 949ms/step - loss: 0.3215 - val_loss: 0.4439 - learning_rate: 0.0010
Epoch 9/100
1014/1014 - 954s - 940ms/step - loss: 0.3151 - val_loss: 0.4443 - learning_rate: 0.0010
Epoch 10/100
1014/1014 - 942s - 929ms/step - loss: 0.3083 - val_loss: 0.4313 - learning_rate: 0.0010

TypeError: Object of type PosixPath is not JSON serializable